## Moirai without covariates

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [1]:
!pip install "numpy<2.0" "pandas<2.2" --force-reinstall -q
!pip uninstall -y torchvision

!pip install "git+https://github.com/SalesforceAIResearch/uni2ts.git" gluonts pyarrow scikit-learn huggingface_hub -q

print("Restart kernel before running script further")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
nilearn 0.13.1 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26

## Config

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from uni2ts.model.moirai import MoiraiFcast, MoiraiModule
from gluonts.dataset.common import ListDataset
from gluonts.evaluation import make_evaluation_predictions
from sklearn.metrics import mean_squared_error

PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH


LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Model

In [ ]:
CSV_OUT = "moirai_eeg_micro_eval_results.csv"
SIZE = "base" # small, base, large
print(f"Loading MOIRAI ({SIZE})...")

model = MoiraiFcast.from_pretrained(
    f"Salesforce/moirai-1.0-R-{SIZE}",
    prediction_length=HORIZON_LENGTH,
    context_length=CONTEXT_LENGTH,
    patch_size="auto",
    num_samples=20,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
    map_location="cuda" if torch.cuda.is_available() else "cpu"
)
print("MOIRAI ready.")

def forecast_windows(model, y_target, target_ch_name, starts, ctx_len, hor_len):
    preds_all = []
    for s in starts:
        ctx_target = y_target[s : s + ctx_len]
        
        test_ds = ListDataset([
            {"target": ctx_target, "start": pd.Timestamp("2024-01-01 00:00:00"), "item_id": "EEG"}
        ], freq="S")
        
        forecast_it, _ = make_evaluation_predictions(
            dataset=test_ds,
            predictor=model,
            num_samples=20
        )
        
        forecasts = list(forecast_it)
        preds = np.median(forecasts[0].samples, axis=0)
        preds_all.append(preds)
        
    return np.array(preds_all)


## Processing functions

In [ ]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [ ]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        y_target = series_to_numpy(row[ch])
        
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        # --- Unified forecast call ---
        preds = forecast_windows(
            model=model, 
            y_target=y_target, 
            target_ch_name=ch, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH
        )
        
        mses_win = []
        for i, s in enumerate(starts):
            tgt_true = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_true, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20))
except ImportError:
    print(df_out.tail(20).to_string())